# VTIA: reentrenar YOLO con MIO-TCD en Colab

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.

Lo que pasa en este cuaderno:

1. Se clona el repositorio y se instalan las dependencias.
2. Se descarga MIO-TCD desde Roboflow y se traduce a las 6 clases de VTIA.
3. Se entrena YOLO26n. Los pesos se guardan en Drive cada época, así que si Colab se desconecta no se pierden.
4. (Opcional) Se evalúan los pesos en la prueba fija propia, con los clips **anonimizados** que estén en Drive.

**Secretos** (icono de la llave, a la izquierda), con acceso del cuaderno activado:

- `ROBOFLOW_API_KEY`: obligatorio. Está en Roboflow → Settings → API Keys.
- `GITHUB_TOKEN`: solo si el repositorio es privado (token de solo lectura).

**Licencia:** MIO-TCD es CC BY-NC-SA 4.0. Uso no comercial, hay que citar a Luo et al. (IEEE TIP 2018), y los pesos que salgan heredan esas condiciones.

## 1. Parámetros

In [ ]:
BRANCH = "feature/comparacion-modelos"   # rama del repositorio con el soporte de MIO-TCD
RF_WORKSPACE = "thesis-ltv3y"
RF_PROJECT = "mio_tcd-f3kk4"
RF_VERSION = 1            # número de versión en la página de Roboflow (pestaña Versions)
RUN_NAME = "mio_tcd"
WEIGHTS = "yolo26n.pt"    # pesos COCO de partida
EPOCHS = 30               # MIO-TCD es grande: en una T4 cada época tarda bastante; se puede reanudar
BATCH = 32
IMGSZ = 640
DRIVE_DIR = "/content/drive/MyDrive/VTIA"   # aquí quedan pesos y resultados

## 2. Drive, código y dependencias

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, subprocess
from pathlib import Path
REPO = "/content/vtia"
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
auth = f"{token}@" if token else ""
url = f"https://{auth}github.com/alejodiazzz/Deteccion-y-Analisis-de-trafico-mediante-IA.git"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
%cd {REPO}

In [ ]:
# torch/torchvision ya vienen con CUDA en Colab; requirements.txt no los fija.
# pandas, streamlit y altair son solo del tablero (app/): se omiten para no reemplazar
# el pandas que Colab necesita (google-colab exige pandas 2.2.x).
SKIP = ("pandas", "streamlit", "altair")
reqs = [l for l in Path("requirements.txt").read_text().splitlines() if not l.strip().startswith(SKIP)]
Path("/content/requirements-colab.txt").write_text("\n".join(reqs))
!pip install -q -r /content/requirements-colab.txt roboflow
import torch
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NINGUNA (cambia el entorno a GPU)")

## 3. Descargar MIO-TCD y traducir las clases

In [ ]:
from pathlib import Path
from roboflow import Roboflow

SRC = Path("/content/data/mio_tcd")   # disco local de Colab: mucho más rápido que Drive
if not (SRC / "data.yaml").exists():
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download("yolov8", location=str(SRC))
print((SRC / "data.yaml").read_text())

In [ ]:
from vtia.adapters.mio_tcd import convert_export

RUN = Path("outputs/train") / RUN_NAME
data, counts = convert_export(SRC, RUN / "dataset")
for split in ("train", "val", "test"):
    print(split, {cls: n for (s, cls), n in sorted(counts.items()) if s == split})

## 4. Entrenar

La carpeta `weights/` del entrenamiento apunta a Drive, así que `last.pt` y `best.pt` se guardan allí en cada época.
Si Colab se desconecta, vuelve a correr las secciones 1 a 3, la primera celda de esta sección y luego **4b** (no 4a).

In [ ]:
DRIVE_RUN = Path(DRIVE_DIR) / "train" / RUN_NAME
(DRIVE_RUN / "weights").mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
if not (RUN / "weights").is_symlink():
    os.symlink(DRIVE_RUN / "weights", RUN / "weights", target_is_directory=True)
print("pesos en:", DRIVE_RUN / "weights")

In [ ]:
# 4a. Entrenamiento nuevo
from vtia.adapters.yolo_training import train
best = train(data, WEIGHTS, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, name=RUN_NAME)
print("mejores pesos:", best)

In [ ]:
# 4b. Reanudar tras una desconexión (no correr 4a)
from ultralytics import YOLO
model = YOLO(str(RUN / "weights" / "last.pt"))
model.train(resume=True)

In [ ]:
# Copiar gráficas y results.csv a Drive (los pesos ya están ahí)
import shutil
for f in RUN.iterdir():
    if f.is_file():
        shutil.copy2(f, DRIVE_RUN / f.name)
print(sorted(p.name for p in DRIVE_RUN.iterdir()))

## 5. (Opcional) Evaluar en la prueba fija propia

El mAP de la celda anterior se mide en la validación de MIO-TCD, que son cámaras de Canadá y EE. UU.
El resultado que cuenta es el de sus propios clips. Suba a Drive **solo material anonimizado**:
`annotations/` (zips CVAT MOT 1.1), `anon/` (videos) y `splits.csv`.

In [ ]:
D = Path(DRIVE_DIR) / "data"
best_pt = RUN / "weights" / "best.pt"
!python -m vtia.cli evaluate --cvat {D}/annotations --videos {D}/anon --splits {D}/splits.csv --partition test --models {best_pt} yolo26n.pt
!cp -r outputs/eval {DRIVE_DIR}/